In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder

# Load embeddings and labels
faces_embeddings = np.load("faces_embeddings_done_4classes.npz")
X = faces_embeddings['arr_0']  # Face embeddings (features)
Y = faces_embeddings['arr_1']  # Labels (names)

# Encode labels
encoder = LabelEncoder()
Y_encoded = encoder.fit_transform(Y)

# Define CNN model (Fully Connected Neural Network)
cnn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),  # First hidden layer
    Dropout(0.5),  # Prevents overfitting
    Dense(64, activation='relu'),  # Second hidden layer
    Dense(len(np.unique(Y)), activation='softmax')  # Output layer
])

# Compile the CNN model
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the CNN model
cnn_model.fit(X, Y_encoded, epochs=50, batch_size=12, validation_split=0.2)

# Save the trained CNN model
cnn_model.save("face_recognition_cnn.h5")

# Save the label encoder
import pickle
with open("encoder.pkl", 'wb') as file:
    pickle.dump(encoder, file)


In [ ]:
print("Predicted index:", predicted_class)
print("Decoded name:", final_name)
print("Encoder classes:", encoder.classes_)


In [1]:
import cv2 as cv
import numpy as np
import pickle
from keras_facenet import FaceNet
from tensorflow.keras.models import load_model

# Load the trained CNN model
cnn_model = load_model("face_recognition_cnn.h5")

# Load the label encoder
encoder = pickle.load(open("encoder.pkl", 'rb'))

# Initialize FaceNet and Haar cascade
facenet = FaceNet()
haarcascade = cv.CascadeClassifier(cv.data.haarcascades + "haarcascade_frontalface_default.xml")

# Start webcam
cap = cv.VideoCapture(0)

while cap.isOpened():
    _, frame = cap.read()
    rgb_img = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    gray_img = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    faces = haarcascade.detectMultiScale(gray_img, 1.3, 5)

    # Correct indentation of the loop
    for x, y, w, h in faces:
        img = rgb_img[y:y + h, x:x + w]
        img = cv.resize(img, (160, 160))  # Resize to 160x160
        img = np.expand_dims(img, axis=0)
        ypred = facenet.embeddings(img)

        # Predict using CNN
        probabilities = cnn_model.predict(ypred)
        predicted_class = np.argmax(probabilities, axis=1)[0]
        final_name = encoder.inverse_transform(np.array([predicted_class]))[0]

        # Draw the rectangle and label
        cv.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
        cv.putText(frame, str(final_name), (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA)

    # Display the video frame
    cv.imshow("Face Recognition", frame)
    
    if cv.waitKey(1) & 0xFF == ord('q'):  # Use 0xFF to avoid errors
        break

cap.release()
cv.destroyAllWindows()


ModuleNotFoundError: No module named 'tensorflow.keras'

In [6]:
import cv2 as cv
import numpy as np
import pickle
from keras_facenet import FaceNet
from tensorflow.keras.models import load_model

# Load the trained CNN model
cnn_model = load_model("face_recognition_cnn.h5")

# Load the label encoder
encoder = pickle.load(open("encoder.pkl", 'rb'))

# Initialize FaceNet and Haar cascade
facenet = FaceNet()
haarcascade = cv.CascadeClassifier(cv.data.haarcascades + "haarcascade_frontalface_default.xml")

# Confidence threshold
THRESHOLD = 0.6  # Adjust based on model performance

# Start webcam
cap = cv.VideoCapture(0)

while cap.isOpened():
    _, frame = cap.read()
    rgb_img = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    gray_img = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    faces = haarcascade.detectMultiScale(gray_img, 1.3, 5)

    for x, y, w, h in faces:
        img = rgb_img[y:y + h, x:x + w]
        img = cv.resize(img, (160, 160))  # Resize to 160x160
        img = np.expand_dims(img, axis=0)
        ypred = facenet.embeddings(img)

        # Predict using CNN
        probabilities = cnn_model.predict(ypred)
        max_prob = np.max(probabilities)
        predicted_class = np.argmax(probabilities, axis=1)[0]

        # Set name based on confidence threshold
        if max_prob < THRESHOLD:
            final_name = "Unknown"
        else:
            final_name = encoder.inverse_transform(np.array([predicted_class]))[0]

        # Draw the rectangle and label
        cv.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
        cv.putText(frame, final_name, (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA)

    # Display the video frame
    cv.imshow("Face Recognition", frame)
    
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()


TypeError: Error when deserializing class 'InputLayer' using config={'batch_shape': [None, 512], 'dtype': 'float32', 'sparse': False, 'name': 'input_layer'}.

Exception encountered: Unrecognized keyword arguments: ['batch_shape']

In [7]:
import tensorflow as tf
print(tf.__version__)


2.15.0
